# LLM Evaluation, Benchmarking, and Integration
## Week 8 — Day 4 | DI GenAI & Machine Learning Bootcamp 2026

**Goal:** Systematically evaluate and compare summarization models using accuracy and ROUGE metrics.

**Models compared:** `t5-small` · `t5-base` · `gpt2`  
**Metrics:** Exact-match accuracy · ROUGE-1 · ROUGE-2 · ROUGE-L · ROUGE-Lsum

**Workflow:**
```
Dataset (news articles + reference summaries)
      ↓
Generate summaries with T5 / GPT-2
      ↓
Evaluate: accuracy (exact match)  →  almost always 0%
          ROUGE-N (n-gram overlap) →  meaningful signal
      ↓
Compare models side-by-side
```

## Part I — Setup

In [ ]:
!pip install rouge_score==0.1.2 -q
!pip install evaluate -q
!pip install -U accelerate -q
!pip install datasets -q
!pip install nltk -q
print("Libraries installed ✓")

In [ ]:
import gc
import os
import warnings
import pandas as pd
import numpy as np
import torch
import evaluate
import nltk
from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,
    GPT2LMHeadModel,
    GPT2Tokenizer,
)
from IPython.display import display

warnings.filterwarnings("ignore")

nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print("All imports OK ✓")

## Part II — Dataset Loading and Exploration

In [ ]:
# Load train.csv / test.csv if present; otherwise fall back to CNN/DailyMail from HuggingFace
def load_summarization_data():
    if os.path.exists("train.csv") and os.path.exists("test.csv"):
        print("Loading local train.csv / test.csv…")
        train_df = pd.read_csv("train.csv")
        test_df  = pd.read_csv("test.csv")
        # Standardise column names
        for df in (train_df, test_df):
            if "prompt_text" not in df.columns and "article" in df.columns:
                df.rename(columns={"article": "prompt_text",
                                   "highlights": "prompt_title"}, inplace=True)
    else:
        print("train.csv not found — loading CNN/DailyMail from Hugging Face…")
        from datasets import load_dataset
        hf_train = load_dataset("cnn_dailymail", "3.0.0", split="train[:2000]")
        hf_test  = load_dataset("cnn_dailymail", "3.0.0", split="test[:500]")
        train_df = pd.DataFrame({
            "prompt_text" : hf_train["article"],
            "prompt_title": hf_train["highlights"],
        })
        test_df = pd.DataFrame({
            "prompt_text" : hf_test["article"],
            "prompt_title": hf_test["highlights"],
        })
        print("CNN/DailyMail loaded ✓  (using 'article' → prompt_text, 'highlights' → prompt_title)")
    return train_df, test_df


train_df, test_df = load_summarization_data()
print(f"\nTrain shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
print(f"Columns     : {list(train_df.columns)}")

In [ ]:
# Sample smaller subsets to reduce computational load
train_sample = train_df.sample(n=100, random_state=42).reset_index(drop=True)
test_sample  = test_df.sample(n=50,  random_state=42).reset_index(drop=True)

print(f"Train sample : {len(train_sample)} rows")
print(f"Test  sample : {len(test_sample)} rows")

In [ ]:
# Display first example: article (prompt_text) + reference summary (prompt_title)
print("=== First training example ===")
print(f"\n[prompt_text — Article]")
print(train_sample["prompt_text"].iloc[0][:600], "…")
print(f"\n[prompt_title — Reference Summary]")
print(train_sample["prompt_title"].iloc[0])

In [ ]:
# Inspect the sampled DataFrames
print("=== Training Sample ===")
display(train_sample[["prompt_text", "prompt_title"]].head(5))

print("\n=== Test Sample ===")
display(test_sample[["prompt_text", "prompt_title"]].head(5))

# Text length statistics
print("\n=== Article length statistics (train sample) ===")
print(train_sample["prompt_text"].str.len().describe().to_string())
print("\n=== Summary length statistics (train sample) ===")
print(train_sample["prompt_title"].str.len().describe().to_string())

## Part III — Summarization with T5

T5 (Text-to-Text Transfer Transformer) is an encoder-decoder model trained to convert any NLP task into a text-to-text format. Prepending `"summarize: "` instructs it to produce a summary.

In [ ]:
def batch_generator(data: list, batch_size: int):
    """Yield successive batches of `batch_size` from `data`."""
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]


def summarize_with_t5(
    articles:   list[str],
    model_name: str  = "t5-small",
    batch_size: int  = 4,
    max_input:  int  = 512,
    max_output: int  = 128,
) -> list[str]:
    """
    Generate abstractive summaries for a list of articles using a T5 model.

    Parameters
    ----------
    articles   : raw article strings
    model_name : HuggingFace model id (e.g. 't5-small', 't5-base')
    batch_size : articles per forward pass
    max_input  : max tokens for input
    max_output : max tokens for generated summary

    Returns
    -------
    List of decoded summary strings.
    """
    print(f"Loading {model_name}…")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = T5ForConditionalGeneration.from_pretrained(model_name).to(DEVICE)
    model.eval()

    summaries = []

    for batch in batch_generator(articles, batch_size):
        # Prepend "summarize: " instruction prefix
        prefixed = [f"summarize: {a}" for a in batch]

        inputs = tokenizer(
            prefixed,
            return_tensors = "pt",
            max_length     = max_input,
            truncation     = True,
            padding        = True,
        ).to(DEVICE)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids      = inputs["input_ids"],
                attention_mask = inputs["attention_mask"],
                max_new_tokens = max_output,
                num_beams      = 4,
                early_stopping = True,
            )

        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        summaries.extend(decoded)

        # Free GPU/CPU memory after each batch
        del inputs, output_ids
        torch.cuda.empty_cache()
        gc.collect()

    del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"{model_name} summaries generated ✓  ({len(summaries)} total)")
    return summaries


print("summarize_with_t5() and batch_generator() defined ✓")

In [ ]:
# Generate summaries for the training sample using t5-small
articles_train = train_sample["prompt_text"].tolist()

t5_small_summaries = summarize_with_t5(
    articles   = articles_train,
    model_name = "t5-small",
    batch_size = 4,
)

In [ ]:
# Display generated summaries alongside reference summaries
results_df = pd.DataFrame({
    "reference_summary" : train_sample["prompt_title"].tolist(),
    "t5_small_summary"  : t5_small_summaries,
})

print("=== Generated vs Reference Summaries (t5-small) ===")
display(results_df.head(10))

# Side-by-side for the first 3 examples
print("\n=== Detailed comparison (first 3 examples) ===")
for i in range(3):
    print(f"\n[Example {i+1}]")
    print(f"  Reference : {results_df['reference_summary'].iloc[i][:200]}")
    print(f"  t5-small  : {results_df['t5_small_summary'].iloc[i][:200]}")

## Part IV — Accuracy Evaluation

Exact-match accuracy counts how many generated summaries are **character-for-character identical** to the reference.

In [ ]:
def calculate_accuracy(
    references: list[str],
    predictions: list[str],
) -> float:
    """Exact-match accuracy: fraction of predictions identical to the reference."""
    matches = sum(
        ref.strip().lower() == pred.strip().lower()
        for ref, pred in zip(references, predictions)
    )
    return matches / len(references)


references  = train_sample["prompt_title"].tolist()
predictions = t5_small_summaries

accuracy = calculate_accuracy(references, predictions)
print(f"Exact-match accuracy (t5-small) : {accuracy:.4f}  ({accuracy*100:.2f}%)")

print("""
Why is accuracy ≈ 0%?
─────────────────────
Exact-match accuracy requires the generated text to be *identical* to the reference.
In abstractive summarization, the model paraphrases — it uses different words to convey
the same meaning. Even a single extra comma or different synonym makes the match fail.

This illustrates why exact-match is the WRONG metric for generation tasks.
We need an overlap-based metric that rewards partial similarity → ROUGE.
""")

## Part V — ROUGE Metric Implementation

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures n-gram overlap between generated and reference summaries.

- **ROUGE-1** — unigram overlap (single words)
- **ROUGE-2** — bigram overlap (word pairs)
- **ROUGE-L** — longest common subsequence
- **ROUGE-Lsum** — ROUGE-L computed sentence-by-sentence (better for multi-sentence summaries)

In [ ]:
# Load the ROUGE metric from HuggingFace evaluate
rouge_metric = evaluate.load("rouge")
print("ROUGE metric loaded ✓")

In [ ]:
def compute_rouge_score(
    predictions: list[str],
    references:  list[str],
    use_stemmer: bool = True,
) -> dict:
    """
    Compute ROUGE scores between predictions and references.

    Preprocessing:
    - Each summary is split into sentences with NLTK.
    - Sentences are joined with '\n' so ROUGE-Lsum operates sentence-by-sentence.

    Parameters
    ----------
    predictions : list of generated summary strings
    references  : list of reference summary strings
    use_stemmer : if True, stem words before n-gram comparison

    Returns
    -------
    dict with keys rouge1, rouge2, rougeL, rougeLsum (each 0–1 F1 score)
    """
    def add_newlines(text: str) -> str:
        """Join sentences with newlines for ROUGE-Lsum."""
        sentences = nltk.sent_tokenize(text.strip())
        return "\n".join(sentences)

    preds_fmt = [add_newlines(p) for p in predictions]
    refs_fmt  = [add_newlines(r) for r in references]

    scores = rouge_metric.compute(
        predictions = preds_fmt,
        references  = refs_fmt,
        use_stemmer = use_stemmer,
    )
    return scores


print("compute_rouge_score() defined ✓")

In [ ]:
# Compute ROUGE for t5-small
rouge_t5_small = compute_rouge_score(t5_small_summaries, references)

print("=== ROUGE Scores — t5-small ===")
for k, v in rouge_t5_small.items():
    print(f"  {k:<12} : {v:.4f}  ({v*100:.2f}%)")

## Part VI — Understanding ROUGE Scores

We run controlled experiments to build intuition for what ROUGE scores mean.

In [ ]:
# ── Test 1: Exact match → ROUGE should be 1.0 ─────────────────────
refs_exact  = ["The quick brown fox jumps over the lazy dog."]
preds_exact = ["The quick brown fox jumps over the lazy dog."]

score_exact = compute_rouge_score(preds_exact, refs_exact)
print("=== Test 1: Exact match ===")
print(f"  Reference  : {refs_exact[0]}")
print(f"  Prediction : {preds_exact[0]}")
for k, v in score_exact.items():
    print(f"  {k:<12} : {v:.4f}")
print("  Expected: all scores ≈ 1.0 ✓")

In [ ]:
# ── Test 2: Empty prediction → ROUGE should be 0.0 ────────────────
refs_null  = ["The quick brown fox jumps over the lazy dog."]
preds_null = [""]   # no prediction

score_null = compute_rouge_score(preds_null, refs_null)
print("=== Test 2: Empty prediction ===")
print(f"  Reference  : {refs_null[0]}")
print(f"  Prediction : (empty)")
for k, v in score_null.items():
    print(f"  {k:<12} : {v:.4f}")
print("  Expected: all scores ≈ 0.0 ✓")

In [ ]:
# ── Test 3: Stemming effect ────────────────────────────────────────
refs_stem  = ["The cats are running quickly."]
preds_stem = ["The cat ran quickly."]

score_no_stem  = compute_rouge_score(preds_stem, refs_stem, use_stemmer=False)
score_with_stem = compute_rouge_score(preds_stem, refs_stem, use_stemmer=True)

print("=== Test 3: Effect of stemming ===")
print(f"  Reference      : {refs_stem[0]}")
print(f"  Prediction     : {preds_stem[0]}")
print(f"  ({'without stemmer':>18})  ({'with stemmer':>18})")
for k in score_no_stem:
    print(f"  {k:<12} : {score_no_stem[k]:.4f}   {'→':5}   {score_with_stem[k]:.4f}")
print("  Insight: Stemming merges 'cats'→'cat' and 'running'→'run', so overlap increases.")

In [ ]:
# ── Test 4: N-gram analysis — varying overlap ──────────────────────
print("=== Test 4: ROUGE-1 vs ROUGE-2 with increasing overlap ===\n")

ref = "The cat sat on the mat and looked at the rat."
cases = [
    ("No overlap",         "The dog ran in the park on a sunny day."),
    ("Partial overlap",    "The cat sat on the floor and looked around."),
    ("High overlap",       "The cat sat on the mat and looked at the big rat."),
    ("Near-exact",         "The cat sat on the mat and looked at the rat today."),
]

print(f"Reference: {ref}\n")
print(f"{'Case':<20} {'ROUGE-1':>10} {'ROUGE-2':>10} {'ROUGE-L':>10}")
print("-" * 55)
for label, pred in cases:
    s = compute_rouge_score([pred], [ref])
    print(f"{label:<20} {s['rouge1']:>10.4f} {s['rouge2']:>10.4f} {s['rougeL']:>10.4f}")

In [ ]:
# ── Test 5: Symmetry — swapping predictions and references ────────
ref5  = ["The scientist discovered a new exoplanet in the Milky Way."]
pred5 = ["A new exoplanet was found by the scientist in our galaxy."]

score_forward  = compute_rouge_score(pred5, ref5)
score_backward = compute_rouge_score(ref5,  pred5)   # swap pred ↔ ref

print("=== Test 5: Symmetry of ROUGE (pred ↔ ref swapped) ===")
print(f"  Ref  : {ref5[0]}")
print(f"  Pred : {pred5[0]}")
print(f"\n  {'Metric':<12} {'pred→ref':>12} {'ref→pred':>12} {'diff':>10}")
print("-" * 50)
for k in score_forward:
    fwd = score_forward[k]
    bwd = score_backward[k]
    print(f"  {k:<12} {fwd:>12.4f} {bwd:>12.4f} {abs(fwd-bwd):>10.4f}")
print("\n  Insight: ROUGE-N F1 is symmetric; ROUGE-L may differ slightly.")

## Part VII — Comparing Small and Large Models

We compare `t5-small` (60M params), `t5-base` (220M params), and `gpt2` (117M params).

In [ ]:
def summarize_with_gpt2(
    articles:   list[str],
    model_name: str = "gpt2",
    batch_size: int = 4,
    max_article_tokens: int = 800,
    max_new_tokens:     int = 100,
) -> list[str]:
    """
    Generate summaries using GPT-2 with a 'TL;DR:' prompt.

    GPT-2 is a causal (left-to-right) LM — it continues text rather than
    producing a structured output. We append 'TL;DR:' to prime it to generate
    a short summary continuation.

    Token limit: GPT-2 max context = 1024 tokens.
    We truncate input articles to `max_article_tokens` to leave room for output.
    """
    print(f"Loading {model_name}…")
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token
    model = GPT2LMHeadModel.from_pretrained(model_name).to(DEVICE)
    model.eval()

    summaries = []

    for batch in batch_generator(articles, batch_size):
        prompts = []
        for article in batch:
            # Truncate article to keep total tokens within GPT-2's 1024-token limit
            article_tokens = tokenizer.encode(article, truncation=True,
                                              max_length=max_article_tokens)
            truncated      = tokenizer.decode(article_tokens)
            prompts.append(f"{truncated}\n\nTL;DR:")

        inputs = tokenizer(
            prompts,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = max_article_tokens + 10,
        ).to(DEVICE)

        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            output_ids = model.generate(
                input_ids      = inputs["input_ids"],
                attention_mask = inputs["attention_mask"],
                max_new_tokens = max_new_tokens,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
                eos_token_id   = tokenizer.eos_token_id,
            )

        # Decode only the newly generated tokens (skip the input prefix)
        for ids in output_ids:
            new_ids = ids[input_len:]
            decoded = tokenizer.decode(new_ids, skip_special_tokens=True)
            # Keep only the first sentence of the continuation
            first_sentence = decoded.split(".")[0].strip()
            summaries.append(first_sentence if first_sentence else decoded[:200])

        del inputs, output_ids
        torch.cuda.empty_cache()
        gc.collect()

    del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"{model_name} summaries generated ✓  ({len(summaries)} total)")
    return summaries


print("summarize_with_gpt2() defined ✓")

In [ ]:
# Generate summaries with all three models
# (t5-small summaries already computed — reuse them)
t5_base_summaries = summarize_with_t5(
    articles   = articles_train,
    model_name = "t5-base",
    batch_size = 4,
)

gpt2_summaries = summarize_with_gpt2(
    articles   = articles_train,
    model_name = "gpt2",
    batch_size = 4,
)

In [ ]:
def compute_rouge_per_row(
    model_name:  str,
    predictions: list[str],
    references:  list[str],
) -> pd.DataFrame:
    """
    Compute ROUGE scores for each individual (prediction, reference) pair.
    Returns a DataFrame with one row per article and columns for each ROUGE metric.
    """
    rows = []
    for pred, ref in zip(predictions, references):
        score = compute_rouge_score([pred], [ref])
        rows.append({
            "model"      : model_name,
            "prediction" : pred[:120],
            "reference"  : ref[:120],
            "rouge1"     : score["rouge1"],
            "rouge2"     : score["rouge2"],
            "rougeL"     : score["rougeL"],
            "rougeLsum"  : score["rougeLsum"],
        })
    return pd.DataFrame(rows)


print("compute_rouge_per_row() defined ✓")

In [ ]:
# Per-row ROUGE for all three models
df_t5_small = compute_rouge_per_row("t5-small", t5_small_summaries, references)
df_t5_base  = compute_rouge_per_row("t5-base",  t5_base_summaries,  references)
df_gpt2     = compute_rouge_per_row("gpt2",     gpt2_summaries,     references)

print("=== Per-row ROUGE — t5-small (first 5 rows) ===")
display(df_t5_small[["reference", "prediction", "rouge1", "rouge2", "rougeL"]].head())

print("\n=== Per-row ROUGE — t5-base (first 5 rows) ===")
display(df_t5_base[["reference", "prediction", "rouge1", "rouge2", "rougeL"]].head())

print("\n=== Per-row ROUGE — gpt2 (first 5 rows) ===")
display(df_gpt2[["reference", "prediction", "rouge1", "rouge2", "rougeL"]].head())

## Part VIII — Comparing All Models

In [ ]:
def compare_models(
    model_summaries: dict[str, list[str]],
    references:      list[str],
) -> pd.DataFrame:
    """
    Aggregate ROUGE scores for all models into a single comparison DataFrame.
    Each row = one model; columns = avg rouge1, rouge2, rougeL, rougeLsum.
    """
    rows = []
    for model_name, preds in model_summaries.items():
        scores = compute_rouge_score(preds, references)
        rows.append({
            "model"    : model_name,
            "rouge1"   : round(scores["rouge1"],    4),
            "rouge2"   : round(scores["rouge2"],    4),
            "rougeL"   : round(scores["rougeL"],    4),
            "rougeLsum": round(scores["rougeLsum"], 4),
        })
    df = pd.DataFrame(rows).set_index("model")
    return df.sort_values("rouge1", ascending=False)


print("compare_models() defined ✓")

In [ ]:
def compare_models_summaries(
    model_summaries: dict[str, list[str]],
    references:      list[str],
    n_examples:      int = 5,
) -> pd.DataFrame:
    """
    Display generated summaries from all models side-by-side for the first n_examples.
    """
    row_data = {"reference": [r[:120] for r in references[:n_examples]]}
    for model_name, preds in model_summaries.items():
        row_data[model_name] = [p[:120] for p in preds[:n_examples]]
    return pd.DataFrame(row_data)


print("compare_models_summaries() defined ✓")

In [ ]:
# All models dict
all_summaries = {
    "t5-small" : t5_small_summaries,
    "t5-base"  : t5_base_summaries,
    "gpt2"     : gpt2_summaries,
}

# ── Aggregated ROUGE comparison ──
comparison_df = compare_models(all_summaries, references)

print("=== Aggregated ROUGE Scores — All Models ===")
display(comparison_df)

# Bar chart
import matplotlib.pyplot as plt

ax = comparison_df.plot(
    kind    = "bar",
    figsize = (10, 5),
    rot     = 0,
    color   = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"],
)
ax.set_title("Average ROUGE Scores by Model", fontsize=13, fontweight="bold")
ax.set_ylabel("ROUGE Score (F1)")
ax.set_ylim(0, 1)
ax.legend(loc="upper right")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("rouge_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → rouge_comparison.png")

In [ ]:
# ── Side-by-side summary comparison ──
summary_comparison = compare_models_summaries(all_summaries, references, n_examples=5)

print("=== Side-by-Side Summary Comparison (first 5 examples) ===")
display(summary_comparison)

# Detailed print
print("\n=== Detailed view (example 1) ===")
for col in summary_comparison.columns:
    print(f"  [{col}]")
    print(f"    {summary_comparison[col].iloc[0]}")
    print()

In [ ]:
# ── Per-model ROUGE distribution (box plots) ──
all_per_row = pd.concat([df_t5_small, df_t5_base, df_gpt2])

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, metric in zip(axes, ["rouge1", "rouge2", "rougeL", "rougeLsum"]):
    data_by_model = [
        all_per_row[all_per_row["model"] == m][metric].values
        for m in ["t5-small", "t5-base", "gpt2"]
    ]
    ax.boxplot(data_by_model, labels=["t5-small", "t5-base", "gpt2"], patch_artist=True)
    ax.set_title(metric.upper(), fontweight="bold")
    ax.set_ylabel("Score")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.suptitle("Per-Article ROUGE Score Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("rouge_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Distribution plot saved → rouge_distributions.png")

## Summary & Key Takeaways

| Model | Architecture | Params | Strengths for summarisation |
|---|---|---|---|
| `t5-small` | Encoder-decoder | ~60M | Fast, trained for summarization task |
| `t5-base`  | Encoder-decoder | ~220M | Better ROUGE, still CPU-feasible |
| `gpt2`     | Decoder-only    | ~117M | General LM; uses TL;DR trick, no task fine-tuning |

### Metric comparison

| Metric | Measures | Weakness |
|---|---|---|
| Exact accuracy | Identical strings | Always ≈ 0% for generative models |
| ROUGE-1 | Unigram overlap | Ignores word order |
| ROUGE-2 | Bigram overlap | More sensitive to phrasing |
| ROUGE-L | LCS length | Rewards fluent paraphrases |
| ROUGE-Lsum | Sentence-level LCS | Best for multi-sentence summaries |

### Key insights

1. **Exact-match accuracy is useless for generation** — it requires byte-for-byte identity, which never happens in abstractive summarisation.
2. **T5 models outperform GPT-2** on ROUGE because they are trained specifically on the seq2seq summarisation task; GPT-2 is purely autoregressive and just continues text after the TL;DR prompt.
3. **Larger T5 (t5-base) ≥ t5-small** — more parameters → better coverage of reference n-grams.
4. **Stemming helps ROUGE** — it reduces surface-form variation (run/runs/running → run), rewarding semantically equivalent summaries.
5. **ROUGE-2 is harder to score than ROUGE-1** — bigram order must also match; this penalises paraphrases that change word order.